# GibbsQ Final Preflight Check

This notebook performs final readiness checks before running large experiments:
- Config loading/validation
- Hardware backend visibility (CPU/GPU/TPU via JAX)
- Quick debug simulations for NumPy and JAX engines


In [ ]:
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
from omegaconf import OmegaConf

from gibbsq.core.config import hydra_to_config, validate
from gibbsq.core.policies import SoftmaxRouting
from gibbsq.engines.numpy_engine import simulate
from gibbsq.engines.jax_engine import simulate_jax


In [ ]:
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cfg_dir = repo_root / 'configs'

for name in ['default.yaml', 'small.yaml', 'large.yaml']:
    raw = OmegaConf.load(cfg_dir / name)
    cfg = hydra_to_config(raw)
    validate(cfg)
    print(f'✅ {name}: valid | N={cfg.system.num_servers}, lambda={cfg.system.arrival_rate}')


In [ ]:
print('Default backend:', jax.default_backend())
for idx, d in enumerate(jax.devices()):
    print(f'  Device {idx}: platform={d.platform}, kind={getattr(d, "device_kind", "unknown")}')


In [ ]:
# NumPy debug run
res_np = simulate(
    num_servers=2,
    arrival_rate=1.0,
    service_rates=np.array([1.2, 1.4]),
    policy=SoftmaxRouting(alpha=1.0),
    sim_time=50.0,
    sample_interval=1.0,
    rng=np.random.default_rng(0),
)
print('NumPy debug OK:', res_np.arrival_count, res_np.departure_count, res_np.states.shape)


In [ ]:
# JAX debug run
times, states, (arr, dep) = simulate_jax(
    num_servers=2,
    arrival_rate=1.0,
    service_rates=jnp.array([1.2, 1.4]),
    alpha=1.0,
    sim_time=50.0,
    sample_interval=1.0,
    key=jax.random.PRNGKey(0),
    max_samples=60,
)
print('JAX debug OK:', int(arr), int(dep), states.shape)
